In [1]:
# Neural Identifier Training - 2-DOF Planar Manipulator
# Methods: EKF, UKF, Particle Filter
# Con características específicas por neurona

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# 1) True nonlinear system (2-DOF Robot Arm)
# ============================================================
def plant_dynamics(state, u):
    """
    Dynamics of a 2-link planar robot arm.
    state = [q1, q2, dq1, dq2] (Angles and Velocities)
    u     = [tau1, tau2]       (Torques)
    """
    # Robot Parameters
    m1, m2 = 1.0, 1.0  # Mass (kg)
    l1, l2 = 1.0, 1.0  # Lengths (m)
    g = 9.81
    
    q1, q2, dq1, dq2 = state
    tau1, tau2 = u

    # --- Mass Matrix M(q) ---
    c2 = np.cos(q2)
    s2 = np.sin(q2)
    
    M11 = (m1 + m2) * l1**2 + m2 * l2**2 + 2 * m2 * l1 * l2 * c2
    M12 = m2 * l2**2 + m2 * l1 * l2 * c2
    M21 = M12
    M22 = m2 * l2**2
    M = np.array([[M11, M12], [M21, M22]])

    # --- Coriolis/Centrifugal Matrix C(q, dq) ---
    h = -m2 * l1 * l2 * s2
    C11 = h * dq2
    C12 = h * (dq1 + dq2)
    C21 = -h * dq1
    C22 = 0.0
    C = np.array([[C11, C12], [C21, C22]])

    # --- Gravity Vector G(q) ---
    s1 = np.sin(q1)
    s12 = np.sin(q1 + q2)
    G1 = (m1 + m2) * g * l1 * s1 + m2 * g * l2 * s12
    G2 = m2 * g * l2 * s12
    G = np.array([G1, G2])

    # --- Equation of Motion: M*ddq + C*dq + G = tau ---
    damping = 0.5 * np.array([dq1, dq2])
    torque_vector = np.array([tau1, tau2])
    
    rhs = torque_vector - (C @ np.array([dq1, dq2])) - G - damping
    
    # Solve for accelerations
    ddq = np.linalg.solve(M, rhs)
    
    return np.concatenate(([dq1, dq2], ddq))

def plant(x_k, u_k, dt=0.01, process_noise_std=1e-4):
    """
    Euler integration step with Gaussian process noise.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot
    
    # Add Gaussian process noise
    noise = np.random.randn(4) * process_noise_std
    return x_kp1 + noise

# ============================================================
# 2) RHONN structure - CARACTERÍSTICAS POR NEURONA
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z). Beta reduced to widen the active range."""
    z = np.clip(z, -50, 50) 
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input, neuron_index):
    """
    Features for 2-DOF Arm - ESPECÍFICAS PARA CADA NEURONA.
    
    x_est = [q1, q2, dq1, dq2]
    u_input = [tau1, tau2]
    neuron_index: índice de la neurona (0=q1, 1=q2, 2=dq1, 3=dq2)
    
    Cada neurona puede tener su propia estructura de características.
    """
    q1, q2, dq1, dq2 = x_est
    tau1, tau2 = u_input
    
    # Términos básicos sigmoidales
    s_q1 = sigmoidal(q1)
    s_q2 = sigmoidal(q2)
    s_dq1 = sigmoidal(dq1)
    s_dq2 = sigmoidal(dq2)
    
    # Términos trigonométricos (útiles para dinámica de robots)
    sin_q1 = np.sin(q1)
    sin_q2 = np.sin(q2)
    cos_q1 = np.cos(q1)
    cos_q2 = np.cos(q2)
    sin_q1q2 = np.sin(q1 + q2)
    
    # ========== CARACTERÍSTICAS ESPECÍFICAS POR NEURONA ==========
    
    if neuron_index == 0:  # Neurona para q1 (ángulo articulación 1)
        return np.array([
            s_q1,                      # Estado actual
            s_dq1,                     # Velocidad actual
            s_q1 * s_dq1,             # Interacción ángulo-velocidad
            s_q1**2,                   # Término cuadrático
            s_q2,                      # Acoplamiento con q2
            # sin_q1,                    # Término gravitacional
            tau1 * 0.1,               # Entrada de control
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 1:  # Neurona para q2 (ángulo articulación 2)
        return np.array([
            s_q2,                      # Estado actual
            s_dq2,                     # Velocidad actual
            s_q2 * s_dq2,             # Interacción ángulo-velocidad
            s_q2**3,                   # Término cúbico (mayor no linealidad)
            s_q1 * s_q2,              # Acoplamiento con q1
            # sin_q2,                    # Término gravitacional
            # sin_q1q2,                  # Acoplamiento cinemático
            tau2 * 0.1,               # Entrada de control
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 2:  # Neurona para dq1 (velocidad articulación 1)
        return np.array([
            s_dq1,                     # Estado actual
            s_dq1**2,                  # Término cuadrático (fricción)
            s_q1,                      # Dependencia del ángulo
            s_q2 * s_dq1,             # Coriolis proxy
            s_q2 * s_dq2,             # Coriolis proxy
            s_dq2,                     # Acoplamiento velocidades
            # cos_q2,                    # Términos de masa variable
            tau1 * 0.1,               # Entrada de control (lineal)
            tau2 * 0.05,              # Acoplamiento de torques
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 3:  # Neurona for dq2 (velocidad articulación 2)
        return np.array([
            s_dq2,                     # Estado actual
            s_dq2**2,                  # Término cuadrático (fricción)
            s_q2,                      # Dependencia del ángulo
            s_q1 * s_dq2,             # Coriolis proxy
            s_dq1 * s_dq2,            # Interacción velocidades
            s_dq1,                     # Acoplamiento con dq1
            # cos_q2,                    # Términos de masa variable
            tau2 * 0.1,               # Entrada de control (lineal)
            tau1 * 0.05,              # Acoplamiento de torques
            # 1.0                        # Bias
        ])
    
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# Función auxiliar para obtener el tamaño de características de cada neurona
def get_z_size(neuron_index):
    """Retorna el número de características para una neurona dada."""
    if neuron_index == 0:
        return 6
    elif neuron_index == 1:
        return 6
    elif neuron_index == 2:
        return 8
    elif neuron_index == 3:
        return 8
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# ============================================================
# 3) Trainers (EKF, UKF, PF) - ADAPTADOS PARA MÚLTIPLES TAMAÑOS
# ============================================================

class Generic_RHONN_Trainer:
    """ Base class to handle the loop logic easily """
    def get_prediction(self, weights, x_k, u_k, neuron_idx):
        z = construct_z_vector(x_k, u_k, neuron_idx)
        return np.dot(weights, z)

class EKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, P0=1.0, Q=1e-4, R=1e-5):
        self.n_neurons = n_neurons
        # Cada neurona tiene su propio número de pesos
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i))*P0 for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*Q for i in range(n_neurons)]
        self.R = R
        self.eta = eta

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            H = z.reshape(-1, 1)
            
            # Predict P
            P_pred = self.P[i] + self.Q_matrices[i]
            
            # Kalman Gain
            S = self.R + (H.T @ P_pred @ H)[0,0]
            K = (P_pred @ H).flatten() / S
            
            # Error
            y_pred = np.dot(self.weights[i], z)
            err = x_kp1[i] - y_pred
            
            # Update Weights
            self.weights[i] += self.eta * K * err
            
            # Update Covariance (Joseph form)
            I_KH = np.eye(len(z)) - np.outer(K, z)
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K, K)*self.R

class UKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, alpha=1e-2):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i)) for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*1e-4 for i in range(n_neurons)]
        self.R = 1e-5
        self.eta = eta
        self.alpha = alpha

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            n = get_z_size(i)
            
            # Sigma params (calculados para cada neurona)
            lambda_ = self.alpha**2 * n - n
            Wm = np.full(2*n+1, 1/(2*(n+lambda_)))
            Wc = np.copy(Wm)
            Wm[0] = lambda_/(n+lambda_)
            Wc[0] = Wm[0] + (3 - self.alpha**2)
            
            # Generate Sigmas
            try:
                L = np.linalg.cholesky((n + lambda_) * self.P[i])
            except:
                L = np.eye(n) * 0.1
                
            sigmas = np.zeros((2*n+1, n))
            sigmas[0] = self.weights[i]
            for k in range(n):
                sigmas[k+1] = self.weights[i] + L[:,k]
                sigmas[n+k+1] = self.weights[i] - L[:,k]
            
            # Transform
            Y_sigmas = np.dot(sigmas, z)
            y_mean = np.sum(Wm * Y_sigmas)
            
            # Covariances
            Py = np.sum(Wc * (Y_sigmas - y_mean)**2) + self.R
            Pxy = np.zeros(n)
            for k in range(2*n+1):
                Pxy += Wc[k] * (sigmas[k] - self.weights[i]) * (Y_sigmas[k] - y_mean)
                
            # Update
            K = Pxy / Py
            err = x_kp1[i] - y_mean
            self.weights[i] += self.eta * K * err
            self.P[i] -= np.outer(K, K) * Py
            
            # Regularize P
            self.P[i] += np.eye(n)*1e-6

class PF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_particles=1000, Q_std=None, R_std=None):
        self.n_neurons = n_neurons
        self.n_particles = n_particles
        # Cada neurona tiene partículas de diferente dimensión
        self.particles = [np.random.randn(n_particles, get_z_size(i))*0.2 
                         for i in range(n_neurons)]
        self.weights_pf = [np.ones(n_particles)/n_particles for _ in range(n_neurons)]
        
        # Q_std y R_std por neurona (permite ajuste fino por estado)
        if Q_std is None:
            self.Q_std = [0.5] * n_neurons  # Default: mismo valor para todas
        else:
            self.Q_std = Q_std if isinstance(Q_std, list) else [Q_std] * n_neurons
            
        if R_std is None:
            self.R_std = [0.2] * n_neurons  # Default: mismo valor para todas
        else:
            self.R_std = R_std if isinstance(R_std, list) else [R_std] * n_neurons

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            n_weights = get_z_size(i)
            
            # 1. Drift (con Q_std específico por neurona)
            self.particles[i] += np.random.randn(self.n_particles, n_weights) * self.Q_std[i]
            
            # 2. Weight (con R_std específico por neurona)
            preds = self.particles[i] @ z
            err = x_kp1[i] - preds
            likelihood = np.exp(-0.5 * (err/self.R_std[i])**2)
            self.weights_pf[i] *= (likelihood + 1e-300)
            self.weights_pf[i] /= np.sum(self.weights_pf[i])
            
            # 3. Resample
            eff_N = 1.0 / np.sum(self.weights_pf[i]**2)
            if eff_N < self.n_particles/2:
                indices = np.random.choice(self.n_particles, self.n_particles, 
                                         p=self.weights_pf[i])
                self.particles[i] = self.particles[i][indices]
                self.weights_pf[i].fill(1.0/self.n_particles)
                
    def get_estimates(self):
        return [np.average(self.particles[i], axis=0, weights=self.weights_pf[i]) 
                for i in range(self.n_neurons)]

# ============================================================
# 4) Simulation Main Loop
# ============================================================
if __name__ == "__main__":
    np.random.seed(7517)
    n_steps = 1000
    dt = 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    
    n_states = 4
    
    # Noise parameters
    process_noise_std = 1e-4
    measurement_noise_std = 1e-3  # Measurement noise standard deviation
    
    # Init Trainers
    ekf = EKF_Trainer(n_states, eta=1.0)
    ukf = UKF_Trainer(n_states, eta=0.9)
    pf = PF_Trainer(n_states, n_particles=1200)
    
    # Arrays
    x_true = np.zeros((n_steps, 4))
    x_measured = np.zeros((n_steps, 4))  # Noisy measurements
    x_est_ekf = np.zeros((n_steps, 4))
    x_est_ukf = np.zeros((n_steps, 4))
    x_est_pf = np.zeros((n_steps, 4))
    
    # Initial Conditions
    x_true[0] = [-np.pi/2, 0, 0, 0] 
    x_measured[0] = x_true[0]  # No noise at t=0
    x_est_ekf[0] = x_measured[0]
    x_est_ukf[0] = x_measured[0]
    x_est_pf[0]  = x_measured[0]
    
    # Excitation Input
    u_hist = np.zeros((n_steps, 2))
    for k in range(n_steps):
        tau1 = 30.0 * np.sin(2.0 * t[k]) 
        tau2 = 15.0 * np.cos(3.0 * t[k])
        u_hist[k] = [tau1, tau2]

    print("Simulating 2-DOF Manipulator with neuron-specific features...")
    print("\nEstructura de características por neurona:")
    for i in range(n_states):
        print(f"  Neurona {i}: {get_z_size(i)} características")
    print(f"\nNivel de ruido:")
    print(f"  Proceso: σ = {process_noise_std:.2e}")
    print(f"  Medición: σ = {measurement_noise_std:.2e}")
    
    for k in range(n_steps - 1):
        # 1. Physics Step (with Gaussian process noise)
        x_true[k+1] = plant(x_true[k], u_hist[k], dt, process_noise_std)
        
        # 2. Add Gaussian measurement noise
        x_measured[k+1] = x_true[k+1] + np.random.randn(4) * measurement_noise_std
        
        # 3. Identify (using noisy measurements)
        # EKF
        ekf.update(x_measured[k+1], x_measured[k], u_hist[k])
        for i in range(4): 
            z = construct_z_vector(x_measured[k], u_hist[k], i)
            x_est_ekf[k+1, i] = np.dot(ekf.weights[i], z)
            
        # UKF
        ukf.update(x_measured[k+1], x_measured[k], u_hist[k])
        for i in range(4): 
            z = construct_z_vector(x_measured[k], u_hist[k], i)
            x_est_ukf[k+1, i] = np.dot(ukf.weights[i], z)
            
        # PF
        pf.update(x_measured[k+1], x_measured[k], u_hist[k])
        w_pf = pf.get_estimates()
        for i in range(4): 
            z = construct_z_vector(x_measured[k], u_hist[k], i)
            x_est_pf[k+1, i] = np.dot(w_pf[i], z)
            
        if k % 100 == 0: print(f"Step {k}")

    # ============================================================
    # 5) Métricas de Error
    # ============================================================
    
    print("\n" + "="*70)
    print("CÁLCULO DE MÉTRICAS DE ERROR")
    print("="*70)
    
    # Configuración de formato para tesis
    thesis_config = {
        'font_family': 'Computer Modern, serif',
        'font_size': 14,
        'title_font_size': 16,
        'legend_font_size': 12,
        'line_width_true': 2.5,
        'line_width_est': 2.0,
        'plot_width': 1000,
        'plot_height': 500,
        'grid_color': 'rgba(200, 200, 200, 0.3)',
        'grid_width': 0.5
    }
    
    # Nombres de estados para reportes
    state_names = ['q₁', 'q₂', 'dq₁', 'dq₂']
    
    # Diccionarios para almacenar métricas
    metrics = {
        'MSE': {'EKF': [], 'UKF': [], 'PF': []},
        'RMSE': {'EKF': [], 'UKF': [], 'PF': []},
        'MAE': {'EKF': [], 'UKF': [], 'PF': []},
        'NRMSE': {'EKF': [], 'UKF': [], 'PF': []}
    }
    
    # Calcular métricas para cada estado
    for i in range(4):
        # Errores
        err_ekf = x_true[:, i] - x_est_ekf[:, i]
        err_ukf = x_true[:, i] - x_est_ukf[:, i]
        err_pf = x_true[:, i] - x_est_pf[:, i]
        
        # MSE
        mse_ekf = np.mean(err_ekf**2)
        mse_ukf = np.mean(err_ukf**2)
        mse_pf = np.mean(err_pf**2)
        
        # RMSE
        rmse_ekf = np.sqrt(mse_ekf)
        rmse_ukf = np.sqrt(mse_ukf)
        rmse_pf = np.sqrt(mse_pf)
        
        # MAE
        mae_ekf = np.mean(np.abs(err_ekf))
        mae_ukf = np.mean(np.abs(err_ukf))
        mae_pf = np.mean(np.abs(err_pf))
        
        # NRMSE (normalized by range)
        range_true = np.max(x_true[:, i]) - np.min(x_true[:, i])
        nrmse_ekf = rmse_ekf / range_true if range_true > 0 else 0
        nrmse_ukf = rmse_ukf / range_true if range_true > 0 else 0
        nrmse_pf = rmse_pf / range_true if range_true > 0 else 0
        
        # Almacenar
        metrics['MSE']['EKF'].append(mse_ekf)
        metrics['MSE']['UKF'].append(mse_ukf)
        metrics['MSE']['PF'].append(mse_pf)
        
        metrics['RMSE']['EKF'].append(rmse_ekf)
        metrics['RMSE']['UKF'].append(rmse_ukf)
        metrics['RMSE']['PF'].append(rmse_pf)
        
        metrics['MAE']['EKF'].append(mae_ekf)
        metrics['MAE']['UKF'].append(mae_ukf)
        metrics['MAE']['PF'].append(mae_pf)
        
        metrics['NRMSE']['EKF'].append(nrmse_ekf)
        metrics['NRMSE']['UKF'].append(nrmse_ukf)
        metrics['NRMSE']['PF'].append(nrmse_pf)
    
    # Métricas totales (suma de todos los estados)
    total_metrics = {}
    for metric_name in ['MSE', 'RMSE', 'MAE', 'NRMSE']:
        total_metrics[metric_name] = {
            'EKF': np.sum(metrics[metric_name]['EKF']),
            'UKF': np.sum(metrics[metric_name]['UKF']),
            'PF': np.sum(metrics[metric_name]['PF'])
        }
    
    # Imprimir resultados
    print("\n--- MÉTRICAS POR ESTADO ---")
    for i, state_name in enumerate(state_names):
        print(f"\n{state_name}:")
        print(f"  MSE:   EKF={metrics['MSE']['EKF'][i]:.6f}  UKF={metrics['MSE']['UKF'][i]:.6f}  PF={metrics['MSE']['PF'][i]:.6f}")
        print(f"  RMSE:  EKF={metrics['RMSE']['EKF'][i]:.6f}  UKF={metrics['RMSE']['UKF'][i]:.6f}  PF={metrics['RMSE']['PF'][i]:.6f}")
        print(f"  MAE:   EKF={metrics['MAE']['EKF'][i]:.6f}  UKF={metrics['MAE']['UKF'][i]:.6f}  PF={metrics['MAE']['PF'][i]:.6f}")
        print(f"  NRMSE: EKF={metrics['NRMSE']['EKF'][i]:.4f}  UKF={metrics['NRMSE']['UKF'][i]:.4f}  PF={metrics['NRMSE']['PF'][i]:.4f}")
    
    print("\n--- MÉTRICAS TOTALES (SUMA) ---")
    for metric_name in ['MSE', 'RMSE', 'MAE', 'NRMSE']:
        print(f"\n{metric_name}:")
        print(f"  EKF: {total_metrics[metric_name]['EKF']:.6f}")
        print(f"  UKF: {total_metrics[metric_name]['UKF']:.6f}")
        print(f"  PF:  {total_metrics[metric_name]['PF']:.6f}")
        
        # Determinar mejor filtro
        best_filter = min(total_metrics[metric_name], key=total_metrics[metric_name].get)
        print(f"  🏆 MEJOR: {best_filter}")
    
    # ============================================================
    # 6) Visualización - Formato Tesis
    # ============================================================
    
    print("\nGenerando visualizaciones...")
    
    # --- Gráficas por Estado ---
    states_info = [
        {'idx': 0, 'var': 'q₁', 'desc': 'Ángulo Articulación 1', 'y_label': 'Ángulo q₁ (rad)'},
        {'idx': 1, 'var': 'q₂', 'desc': 'Ángulo Articulación 2', 'y_label': 'Ángulo q₂ (rad)'},
        {'idx': 2, 'var': 'dq₁', 'desc': 'Velocidad Articulación 1', 'y_label': 'Velocidad dq₁ (rad/s)'},
        {'idx': 3, 'var': 'dq₂', 'desc': 'Velocidad Articulación 2', 'y_label': 'Velocidad dq₂ (rad/s)'}
    ]
    
    for state_info in states_info:
        i = state_info['idx']
        
        fig = go.Figure()
        
        fig.add_trace(go.Scatter(
            x=t, y=x_true[:, i],
            mode='lines',
            name='Estado Real',
            line=dict(color='#000000', width=thesis_config['line_width_true']),
            showlegend=True
        ))
        
        fig.add_trace(go.Scatter(
            x=t, y=x_est_ekf[:, i],
            mode='lines',
            name='EKF-RHONN',
            line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
            showlegend=True
        ))
        
        fig.add_trace(go.Scatter(
            x=t, y=x_est_ukf[:, i],
            mode='lines',
            name='UKF-RHONN',
            line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
            showlegend=True
        ))
        
        fig.add_trace(go.Scatter(
            x=t, y=x_est_pf[:, i],
            mode='lines',
            name='PF-RHONN',
            line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
            showlegend=True
        ))
        
        fig.update_layout(
            title={
                'text': f'Estado {state_info["var"]}: {state_info["desc"]} - Manipulador 2-DOF',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
            },
            xaxis_title='Tiempo (s)',
            yaxis_title=state_info['y_label'],
            xaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            legend=dict(
                x=0.02,
                y=0.98,
                xanchor='left',
                yanchor='top',
                bgcolor='rgba(255, 255, 255, 0.9)',
                bordercolor='black',
                borderwidth=1,
                font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
            ),
            font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
            plot_bgcolor='white',
            paper_bgcolor='white',
            width=thesis_config['plot_width'],
            height=thesis_config['plot_height'],
            margin=dict(l=80, r=40, t=80, b=60)
        )
        
        fig.show()
    
    # --- Gráficas de Métricas (RMSE, MAE, NRMSE) ---
    
    # RMSE por estado
    fig_rmse = go.Figure()
    
    x_pos = np.arange(len(state_names))
    width = 0.25
    
    fig_rmse.add_trace(go.Bar(
        name='EKF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['RMSE']['EKF'],
        marker_color='#1f77b4',
        text=[f'{v:.4f}' for v in metrics['RMSE']['EKF']],
        textposition='outside'
    ))
    
    fig_rmse.add_trace(go.Bar(
        name='UKF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['RMSE']['UKF'],
        marker_color='#2ca02c',
        text=[f'{v:.4f}' for v in metrics['RMSE']['UKF']],
        textposition='outside'
    ))
    
    fig_rmse.add_trace(go.Bar(
        name='PF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['RMSE']['PF'],
        marker_color='#d62728',
        text=[f'{v:.4f}' for v in metrics['RMSE']['PF']],
        textposition='outside'
    ))
    
    fig_rmse.update_layout(
        title={
            'text': 'RMSE por Estado - Manipulador 2-DOF',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Estado',
        yaxis_title='RMSE',
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=100, b=60)
    )
    
    fig_rmse.show()
    
    # MAE por estado
    fig_mae = go.Figure()
    
    fig_mae.add_trace(go.Bar(
        name='EKF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['MAE']['EKF'],
        marker_color='#1f77b4',
        text=[f'{v:.4f}' for v in metrics['MAE']['EKF']],
        textposition='outside'
    ))
    
    fig_mae.add_trace(go.Bar(
        name='UKF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['MAE']['UKF'],
        marker_color='#2ca02c',
        text=[f'{v:.4f}' for v in metrics['MAE']['UKF']],
        textposition='outside'
    ))
    
    fig_mae.add_trace(go.Bar(
        name='PF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['MAE']['PF'],
        marker_color='#d62728',
        text=[f'{v:.4f}' for v in metrics['MAE']['PF']],
        textposition='outside'
    ))
    
    fig_mae.update_layout(
        title={
            'text': 'MAE por Estado - Manipulador 2-DOF',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Estado',
        yaxis_title='MAE',
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=100, b=60)
    )
    
    fig_mae.show()
    
    # NRMSE por estado
    fig_nrmse = go.Figure()
    
    fig_nrmse.add_trace(go.Bar(
        name='EKF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['NRMSE']['EKF'],
        marker_color='#1f77b4',
        text=[f'{v:.4f}' for v in metrics['NRMSE']['EKF']],
        textposition='outside'
    ))
    
    fig_nrmse.add_trace(go.Bar(
        name='UKF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['NRMSE']['UKF'],
        marker_color='#2ca02c',
        text=[f'{v:.4f}' for v in metrics['NRMSE']['UKF']],
        textposition='outside'
    ))
    
    fig_nrmse.add_trace(go.Bar(
        name='PF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['NRMSE']['PF'],
        marker_color='#d62728',
        text=[f'{v:.4f}' for v in metrics['NRMSE']['PF']],
        textposition='outside'
    ))
    
    fig_nrmse.update_layout(
        title={
            'text': 'NRMSE por Estado - Manipulador 2-DOF',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Estado',
        yaxis_title='NRMSE',
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=100, b=60)
    )
    
    fig_nrmse.show()
    
    # --- Comparación de métricas totales ---
    fig_total = go.Figure()
    
    filters = ['EKF-RHONN', 'UKF-RHONN', 'PF-RHONN']
    
    fig_total.add_trace(go.Bar(
        name='RMSE Total',
        x=filters,
        y=[total_metrics['RMSE']['EKF'], total_metrics['RMSE']['UKF'], total_metrics['RMSE']['PF']],
        marker_color='#636EFA',
        text=[f'{v:.4f}' for v in [total_metrics['RMSE']['EKF'], total_metrics['RMSE']['UKF'], total_metrics['RMSE']['PF']]],
        textposition='outside'
    ))
    
    fig_total.add_trace(go.Bar(
        name='MAE Total',
        x=filters,
        y=[total_metrics['MAE']['EKF'], total_metrics['MAE']['UKF'], total_metrics['MAE']['PF']],
        marker_color='#EF553B',
        text=[f'{v:.4f}' for v in [total_metrics['MAE']['EKF'], total_metrics['MAE']['UKF'], total_metrics['MAE']['PF']]],
        textposition='outside'
    ))
    
    fig_total.add_trace(go.Bar(
        name='NRMSE Total',
        x=filters,
        y=[total_metrics['NRMSE']['EKF'], total_metrics['NRMSE']['UKF'], total_metrics['NRMSE']['PF']],
        marker_color='#00CC96',
        text=[f'{v:.4f}' for v in [total_metrics['NRMSE']['EKF'], total_metrics['NRMSE']['UKF'], total_metrics['NRMSE']['PF']]],
        textposition='outside'
    ))
    
    fig_total.update_layout(
        title={
            'text': 'Comparación de Métricas Totales - Manipulador 2-DOF',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tipo de Filtro',
        yaxis_title='Valor de Métrica',
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=100, b=60)
    )
    
    fig_total.show()
    
    print("\n✅ Visualización completa con RMSE, MAE y NRMSE.")


Simulating 2-DOF Manipulator with neuron-specific features...

Estructura de características por neurona:
  Neurona 0: 6 características
  Neurona 1: 6 características
  Neurona 2: 8 características
  Neurona 3: 8 características

Nivel de ruido:
  Proceso: σ = 1.00e-04
  Medición: σ = 1.00e-03
Step 0
Step 100
Step 200
Step 300
Step 400
Step 500
Step 600
Step 700
Step 800
Step 900

CÁLCULO DE MÉTRICAS DE ERROR

--- MÉTRICAS POR ESTADO ---

q₁:
  MSE:   EKF=0.000002  UKF=0.003623  PF=0.000190
  RMSE:  EKF=0.001399  UKF=0.060187  PF=0.013780
  MAE:   EKF=0.001070  UKF=0.045770  PF=0.009300
  NRMSE: EKF=0.0000  UKF=0.0021  PF=0.0005

q₂:
  MSE:   EKF=3.314227  UKF=9.765240  PF=246.600912
  RMSE:  EKF=1.820502  UKF=3.124938  PF=15.703532
  MAE:   EKF=0.292852  UKF=1.135232  PF=6.926486
  NRMSE: EKF=0.0634  UKF=0.1088  PF=0.5468

dq₁:
  MSE:   EKF=0.000097  UKF=0.183081  PF=0.000419
  RMSE:  EKF=0.009870  UKF=0.427880  PF=0.020468
  MAE:   EKF=0.005007  UKF=0.256556  PF=0.013806
  NRMSE: EK


✅ Visualización completa con RMSE, MAE y NRMSE.
